## LLM을 활용한 텍스트 요약

LLM은 언어를 "이해"할 수 있기 때문에, 번역이나 요약과 같은 작업에 적합합니다.

이 노트북에서는 LLM을 사용하여 여러 텍스트, 특히 보험 청구 사례들을 요약해보겠습니다.

### 요구 사항 및 라이브러리 임포트

실습 지침에 따라 올바른 워크벤치 이미지를 선택하여 실행하였다면, 필요한 모든 라이브러리가 이미 설치되어 있을 것입니다.  
그렇지 않은 경우에는 다음 셀의 첫 번째 줄 주석을 해제하여 필요한 패키지를 설치하십시오.

In [ ]:
# 아래 줄은 올바른 워크벤치 이미지를 선택하지 않았거나, 이 노트북을 워크숍 환경 외부에서 사용하는 경우에만 주석을 해제하십시오.
# !pip install --no-cache-dir --no-dependencies --disable-pip-version-check -r requirements.txt

import json
import os
from os import listdir
from os.path import isfile, join

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain_community.llms import VLLMOpenAI

### Langchain 파이프라인

이번에도 Langchain을 사용하여 요약 파이프라인을 정의해보겠습니다.

In [ ]:
# LLM Inference Server URL
inference_server_url = "http://granite-7b-instruct-predictor.ic-shared-llm.svc.cluster.local:8080"

# LLM definition
llm = VLLMOpenAI(           # 우리는 vLLM OpenAI 호환 API 클라이언트를 사용하고 있습니다. 하지만 모델은 OpenAI가 아니라 OpenShift AI에서 실행되고 있습니다.
    openai_api_key="EMPTY",   # 따라서 OpenAI 키가 필요하지 않습니다.
    openai_api_base= f"{inference_server_url}/v1",
    model_name="granite-7b-instruct",
    top_p=0.92,
    temperature=0.01,
    max_tokens=512,
    presence_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

우리가 이번에 사용할 **템플릿**은 요약 작업을 맞게 구성한 것입니다.
프롬프트 내용은 아래와 같습니다.

* 당신은 도움이 되고, 정중하며, 정직한 어시스턴트입니다.
* 항상 신중하고, 존중하며, 진실되게 돕고, 유용하면서도 안전하게 응답해야 합니다.
* 해롭거나, 비윤리적이거나, 편견이 있거나, 부정적인 내용은 피해야 합니다. 답변은 공정성과 긍정성을 증진해야 합니다.
* **제가 텍스트를 제공할 것이며, 당신은 그것을 가능한 한 잘 요약해야 합니다.**

In [ ]:
template="""<|system|>
You are a helpful, respectful and honest assistant.
Always assist with care, respect, and truth. Respond with utmost utility yet securely.
Avoid harmful, unethical, prejudiced, or negative content. Ensure replies promote fairness and positivity.
I will give you a text that you must summarize as best as you can.

<|user|>
### TEXT:
{input}

### SUMMARY:
<|assistant|>
"""
prompt = PromptTemplate(input_variables=["input"], template=template)

이제 모델에 질의할 때 사용할 **conversation** 객체를 생성할 수 있습니다.

In [ ]:
conversation = prompt | llm

이제 모델에 질의할 준비가 완료되었습니다!

`claims` 폴더에는 실제로 수신될 수 있는 보험 청구 예시가 담긴 JSON 파일들이 있습니다.  
이제 해당 파일들을 읽고, 내용을 출력한 뒤, LLM이 생성한 요약 결과도 확인해보겠습니다.

In [ ]:
# 청구 데이터를 읽고 딕셔너리에 저장
claims_path = 'claims'
onlyfiles = [f for f in listdir(claims_path) if isfile(join(claims_path, f))]

claims = {}

for filename in onlyfiles:
    # Opening JSON file
    with open(os.path.join(claims_path, filename), 'r') as file:
        data = json.load(file)
    claims[filename] = data

In [ ]:
for filename in onlyfiles:
    print(f"***************************")
    print(f"* 청구: {filename}")
    print(f"***************************")
    print("원본 내용:")
    print("-----------------")
    print(f"제목: {claims[filename]['subject']}\n내용:\n{claims[filename]['content']}\n\n")
    print('요약:')
    print("--------")
    summary_input = f"Subject: {claims[filename]['subject']}\nContent:\n{claims[filename]['content']}"
    conversation.invoke(input=summary_input);
    print("\n\n                          ----====----\n")

실습을 계속 진행하시다가, 실습의 3.7 챕터에서 이 노트북으로 다시 돌아와서 선택적 실습을 더 진행하실 수도 있습니다.